# forex_strat_006

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (215).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `forex` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 16 |
| Detected functions | resample_1hr_custom, mean_reversion_signal, cy_backtest_trades, backtest_trades, calculate_metrics_fast, main |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Data/Forex_T8_2020_202505.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2023-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['EURUSD', 'USDJPY', 'GBPUSD', 'AUDUSD', 'USDCAD', 'EURGBP',
       'NZDUSD', 'USDCHF']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('7min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np

import numpy as np
import pandas as pd

def mean_reversion_signal(df, smooth_length=172, lyap_window=148, rs_window=24, rs_threshold=0.15):
    df = df.copy()
    close = df['close'].to_numpy(dtype=np.float64)
    high = df['high'].to_numpy(dtype=np.float64)
    low = df['low'].to_numpy(dtype=np.float64)
    open_ = df['open'].to_numpy(dtype=np.float64)
    n = len(close)

    if n < max(smooth_length, lyap_window, rs_window):
        df['Signal'] = 0
        return df

    # --- Step 1: JMA (smoothed close) ---
    alpha = 2.0 / (smooth_length + 1)
    jma = np.empty(n, dtype=np.float64)
    jma[0] = close[0]
    for i in range(1, n):
        jma[i] = jma[i - 1] + (close[i] - jma[i - 1]) * alpha

    # --- Step 2: Lyapunov-style stddev of deviation ---
    deviations = close - jma
    lyap = pd.Series(deviations).rolling(lyap_window, min_periods=1).std().to_numpy(dtype=np.float64)

    # --- Step 3: Rogers-Satchell Volatility (vectorized) ---
    with np.errstate(divide='ignore', invalid='ignore'):
        log_hc = np.log(np.divide(high, close, out=np.zeros_like(close), where=close > 0))
        log_ho = np.log(np.divide(high, open_, out=np.zeros_like(open_), where=open_ > 0))
        log_lc = np.log(np.divide(low, close, out=np.zeros_like(close), where=close > 0))
        log_lo = np.log(np.divide(low, open_, out=np.zeros_like(open_), where=open_ > 0))

    rs = log_hc * log_ho + log_lc * log_lo
    rs_vol = pd.Series(rs).rolling(rs_window, min_periods=1).mean().to_numpy(dtype=np.float64)
    rs_vol = np.sqrt(np.clip(rs_vol, 0, None))  # Ensure non-negative under sqrt

    # --- Step 4: RS Volatility Quantile Filter ---
    vol_cutoff = np.nanquantile(rs_vol, rs_threshold)
    rs_vol_filter = rs_vol < vol_cutoff

    # --- Step 5: Signal Logic (vectorized) ---
    signal = np.zeros(n, dtype=np.int8)
    long_condition = (deviations < -lyap) & rs_vol_filter
    short_condition = (deviations > lyap) & rs_vol_filter
    signal[long_condition] = 1
    signal[short_condition] = 2

    df['Signal'] = signal
    return df

# Assumes `data` is already loaded with: Ticker, Date, open, high, low, close, Signal
data['Date'] = pd.to_datetime(data['Date'])

# Apply your signal logic
data = data.groupby('Ticker', group_keys=False).apply(mean_reversion_signal)

# Shift signals for next-bar execution
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(np.int8)
data

In [ ]:
summary = data['Signal'].value_counts().reset_index()
summary.columns = ['Signal', 'Count']
print("Number of unique exit reasons:", len(summary))
print(summary)

In [ ]:
load_ext cython

In [ ]:
%%cython
import numpy as np
cimport numpy as np

def cy_backtest_trades(
    object[:, :] data,
    str ticker,
    double capital,
    double fee_rate,
    double slippage_rate,
    double sl_pct
):
    cdef:
        int i, n = data.shape[0]
        bint position_open = False
        double entry_price = 0.0
        object entry_time = None
        int direction = 0  # 1 = long, -1 = short
        double quantity = 0.0
        double cumulative_pnl = 0.0
        object last_exit_time = None
        list trades = []

        double sl_price = 0.0
        double exit_price = 0.0
        double open_, high, low, close
        double pnl = 0.0, fee = 0.0, net_pnl = 0.0
        object time
        int signal
        str reason

    for i in range(n):
        time = data[i, 0]
        open_ = <double>data[i, 1]
        high = <double>data[i, 2]
        low = <double>data[i, 3]
        close = <double>data[i, 4]
        signal = <int>data[i, 5]

        # === Exit conditions ===
        if position_open:
            if direction == 1 and low <= sl_price:
                exit_price = sl_price
                reason = "Stop Loss"
            elif direction == -1 and high >= sl_price:
                exit_price = sl_price
                reason = "Stop Loss"
            elif (direction == 1 and signal == 2) or (direction == -1 and signal == 1):
                exit_price = open_ * (1 - slippage_rate) if direction == 1 else open_ * (1 + slippage_rate)
                reason = "Opposite Signal"
            else:
                continue  # Hold position

            # Exit execution
            pnl = (exit_price - entry_price) * quantity * direction
            fee = fee_rate * (entry_price + exit_price) * quantity
            net_pnl = pnl - fee
            cumulative_pnl += net_pnl
            trades.append((
                ticker, entry_time, time, entry_price, exit_price, net_pnl,
                reason, cumulative_pnl, "long" if direction == 1 else "short"
            ))

            # Mark exit
            position_open = False
            last_exit_time = time

            # === Reverse trade if opposite signal ===
            if reason == "Opposite Signal":
                direction = 1 if signal == 1 else -1
                entry_price = open_ * (1 + slippage_rate) if direction == 1 else open_ * (1 - slippage_rate)
                quantity = capital / entry_price
                entry_time = time
                sl_price = entry_price * (1 - sl_pct * direction)
                position_open = True

        # === New Entry ===
        elif not position_open and signal in (1, 2) and time != last_exit_time:
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate) if direction == 1 else open_ * (1 - slippage_rate)
            quantity = capital / entry_price
            entry_time = time
            sl_price = entry_price * (1 - sl_pct * direction)
            position_open = True

    return trades


In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def backtest_trades(
    data: pd.DataFrame,
    sl_pct: float
) -> pd.DataFrame:

    initial_capital = 100000
    fee_rate = 0.0002  # 0.04% round trip
    slippage_rate = 0.0002  # 0.05% round trip

    # Convert 'Date' to datetime and avoid deprecated warning
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()

    # Prepare each ticker's data for cython backtest
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][[
            "Date", "open", "high", "low", "close", "Signal"
        ]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades = []

    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = cy_backtest_trades(
            ticker_map[ticker],
            ticker,
            initial_capital,
            fee_rate,
            slippage_rate,
            sl_pct
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

tradebook = backtest_trades(
    data,
    sl_pct=0.01,            # 0.2% SL
)

tradebook.to_csv("/content/drive/MyDrive/tradebook_US30_2022_25.csv", index=False)
tradebook


In [ ]:
import os
import time
import pickle
import pandas as pd
import numpy as np
from itertools import product
from tqdm import tqdm

# === Hyperparameter Grid ===
param_grid = {
    'smooth_length': [24, 60, 120, 132, 148, 160, 172],
    'lyap_window': [24, 60, 132, 148, 160, 172],
    'rs_window': [24],
    'sl_pct': [0.01, 0.02],
    'rs_threshold': [0.1, 0.15, 0.2]
}
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
print(f"Total combinations: {len(param_combinations)}")

# === Paths ===
csv_path = "/content/drive/MyDrive/hyperparameter_results_5.csv"
cache_path = "/content/drive/MyDrive/hyperparameter_checkpoint_5.pkl"

# === Fast Metrics ===
def calculate_metrics_fast(tradebook):
    if tradebook.empty:
        return 0.0, 0.0, 0.0, 0.0, 0.0, 0
    dates = pd.to_datetime(tradebook['Entry Time'], errors='coerce')
    pnls = tradebook['PnL'].values
    equity = np.cumsum(pnls)
    total_profit = equity[-1]
    roll_max = np.maximum.accumulate(equity)
    drawdown = roll_max - equity
    max_drawdown = np.max(drawdown)
    ratio = total_profit / max_drawdown if max_drawdown > 0 else float('inf') if total_profit > 0 else 0.0
    x = np.arange(len(equity))
    y = equity
    x_mean, y_mean = x.mean(), y.mean()
    ss_tot = np.sum((y - y_mean)**2)
    slope = np.dot(x - x_mean, y - y_mean) / np.dot(x - x_mean, x - x_mean)
    y_pred = slope * (x - x_mean) + y_mean
    ss_res = np.sum((y - y_pred)**2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0
    months = pd.to_datetime(dates).values.astype('datetime64[M]')
    month_pnls = pd.Series(pnls).groupby(months).sum().values
    monthly_consistency = month_pnls.mean() / month_pnls.std() if len(month_pnls) > 1 and month_pnls.std() > 0 else 0.0
    total_trades = len(pnls)
    win_rate = np.mean(pnls > 0)
    return ratio, total_profit, r2, monthly_consistency, win_rate, total_trades

# === Main Tuning Function ===
def main():
    global data  # use your already-loaded DataFrame
    start_index = 0
    results = []

    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            checkpoint = pickle.load(f)
            start_index = checkpoint['last_index'] + 1
            results = checkpoint['results']
            print(f"🔁 Resuming from checkpoint at index {start_index}")

    SAVE_EVERY = 100
    SAVE_INTERVAL = 300  # seconds
    last_save_time = time.time()

    try:
        for idx in tqdm(range(start_index, len(param_combinations)), desc="Hyperparameter Tuning"):
            combo = param_combinations[idx]
            param = dict(zip(keys, combo))

            try:
                temp_data = data.copy()
                temp_data = temp_data.groupby("Ticker", group_keys=False).apply(
                    lambda df: mean_reversion_signal(
                        df,
                        smooth_length=param['smooth_length'],
                        lyap_window=param['lyap_window'],
                        rs_window=param['rs_window'],
                        rs_threshold=param['rs_threshold']
                    )
                )

                temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
                temp_data.dropna(subset=['Signal'], inplace=True)
                temp_data['Signal'] = temp_data['Signal'].astype(int)

                tradebook = backtest_trades(
                    temp_data,
                    sl_pct=param['sl_pct'],
                )

                metrics = calculate_metrics_fast(tradebook)
                result = {
                    **param,
                    'Profit/Drawdown': metrics[0],
                    'Total Profit': metrics[1],
                    'Equity R2': metrics[2],
                    'Monthly Consistency': metrics[3],
                    'Win Rate': metrics[4],
                    'Total Trades': metrics[5],
                }
                result['Composite Score'] = (
                    result['Profit/Drawdown'] * 0.3 +
                    result['Equity R2'] * 0.5 +
                    result['Monthly Consistency'] * 0.2
                )

                results.append(result)
                print(f"✅ {idx}: {result}")

                if (
                    len(results) % SAVE_EVERY == 0 or
                    time.time() - last_save_time > SAVE_INTERVAL or
                    idx == len(param_combinations) - 1
                ):
                    df = pd.DataFrame(results)
                    df.to_csv(csv_path, index=False)
                    with open(cache_path, 'wb') as f:
                        pickle.dump({'last_index': idx, 'results': results}, f)
                    last_save_time = time.time()
                    print(f"💾 Saved checkpoint at index {idx}")

            except Exception as e:
                print(f"❌ Error on combo {idx}: {param} — {e}")
                continue

    finally:
        df = pd.DataFrame(results)
        df.to_csv(csv_path, index=False)
        with open(cache_path, 'wb') as f:
            pickle.dump({'last_index': idx, 'results': results}, f)
        print(f"💾 Final checkpoint saved at index {idx}")

# === Run Tuning ===
if __name__ == "__main__":
    main()
